In [ ]:
import pandas as pd

summary_file = "/media/bahaa/Data/Work/2026/Aeromonas_type_material_23022026/Aeromonas_type_material_23022026/ncbi_dataset/data/data_summary.tsv"
df = pd.read_csv(summary_file, sep="\t")

for i, col in enumerate(df.columns):
    print(i, "->", col)


This to extract the genoems and plasmids please be aware of base_dir and GENUS_ABBR

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path
from Bio import SeqIO
from typing import Optional

# =========================
# 1. Configuration
# =========================
# --- Input paths (reading from a specific location) ---
# Using Path() for consistency. This is where your original assembly folders are.
base_dir = Path("/media/bahaa/Data/Work/2026/Aeromonas_type_material_23022026/Aeromonas_type_material_23022026/ncbi_dataset/data")
summary_file = base_dir / "data_summary.tsv"
# CORRECTED: Added this missing variable definition. This is the directory containing the accession folders (e.g., GCF_...).
accession_data_dir = base_dir 

# --- Output paths (writing to the current directory) ---
# CORRECTED: Using Path() and Path.cwd() for all output paths.
output_dir = Path.cwd() / "raw_data"
chr_dir = output_dir / "chromosomes"
plasmid_dir = output_dir / "plasmids"

# --- Parameters ---
GENUS_ABBR: Optional[str] = "A"  # Set to None to use the full genus name

# =========================
# 2. Helper Functions
# =========================
def sanitize_filename(name: str) -> str:
    """Replaces special characters with underscores for safe filenames."""
    return re.sub(r'[^A-Za-z0-9_.-]', '_', name)

def format_organism_name(organism: str, abbr: Optional[str] = None) -> str:
    """Formats an organism name, using a genus abbreviation if provided."""
    parts = organism.split(" ", 1)
    if abbr and len(parts) == 2:
        genus, rest = parts
        return f"{abbr}_{rest.replace(' ', '_')}"
    return organism.replace(" ", "_")

# =========================
# 3. Main Processing Logic
# =========================
def main():
    """
    Main function to process genomes, separate chromosomes and plasmids,
    and generate a summary report.
    """
    # Create output directories if they don't exist
    print(f"Creating output directories in: {output_dir}")
    chr_dir.mkdir(parents=True, exist_ok=True)
    plasmid_dir.mkdir(parents=True, exist_ok=True)

    # Load metadata
    try:
        df = pd.read_csv(summary_file, sep="\t")
    except FileNotFoundError:
        print(f"❌ ERROR: Metadata file not found at '{summary_file}'")
        return

    # Initialize trackers
    mapping = []
    skipped = []
    chromosome_count = 0
    plasmid_count = 0
    assemblies_found = 0
    used_names = set()

    print("🚀 Starting genome processing...")

    for _, row in df.iterrows():
        accession = row["Assembly Accession"]
        organism = row["Organism Scientific Name"]
        
        # --- Construct a new, unique name for the genome ---
        organism_formatted = format_organism_name(organism, GENUS_ABBR)
        
        strain_raw = str(row.get("Organism Qualifier", ""))
        strain = strain_raw.replace("strain:", "").split(";")[0].strip().replace(" ", "_")
        
        if strain and strain.lower() != "nan" and strain not in organism_formatted:
            new_name = f"{organism_formatted}_{strain}"
        else:
            new_name = organism_formatted
            
        new_name = sanitize_filename(new_name)
        
        # Ensure name uniqueness by appending accession if a duplicate is found
        if new_name in used_names:
            new_name = f"{new_name}_{accession}"
        used_names.add(new_name)

        # --- Find the genomic FASTA file ---
        asm_dir = accession_data_dir / accession
        try:
            fna_file = next(asm_dir.glob("*_genomic.fna"))
            assemblies_found += 1
        except StopIteration:
            print(f"⚠️ Skipping {accession}, file not found in '{asm_dir}'.")
            skipped.append(accession)
            continue
            
        # --- Separate and write chromosome and plasmid records ---
        chr_records, plasmid_records = [], []
        for record in SeqIO.parse(fna_file, "fasta"):
            if "plasmid" in record.description.lower():
                plasmid_records.append(record)
            else:
                chr_records.append(record)

        # Write chromosome file
        if chr_records:
            chr_out_path = chr_dir / f"{new_name}.fna"
            fasta_header = chr_out_path.stem  # Use filename without extension as header
            for rec in chr_records:
                rec.id = fasta_header
                rec.description = ""
            SeqIO.write(chr_records, chr_out_path, "fasta")
            mapping.append([accession, new_name, "chromosome", str(chr_out_path)])
            chromosome_count += 1
            # print(f"✅ Chromosome saved: {chr_out_path.name}") # This can be noisy, optionally comment out

        # Write plasmid file
        if plasmid_records:
            plasmid_out_path = plasmid_dir / f"{new_name}.fna"
            fasta_header = plasmid_out_path.stem
            for rec in plasmid_records:
                rec.id = fasta_header
                rec.description = ""
            SeqIO.write(plasmid_records, plasmid_out_path, "fasta")
            mapping.append([accession, new_name, "plasmid", str(plasmid_out_path)])
            plasmid_count += 1
            # print(f"✅ Plasmid saved: {plasmid_out_path.name}") # This can be noisy, optionally comment out

    # --- Save mapping table and print summary ---
    map_file = output_dir / "accession_to_newnames.tsv"
    pd.DataFrame(mapping, columns=["accession", "new_name", "type", "file_path"]).to_csv(
        map_file, sep="\t", index=False
    )
    print(f"\n📑 Mapping file saved: {map_file}")

    print("\n\n===== 📊 SUMMARY REPORT =====")
    print(f"Total assemblies in metadata : {len(df)}")
    print(f"Assemblies found on disk     : {assemblies_found}")
    print(f"Chromosomes written          : {chromosome_count}")
    print(f"Plasmids written             : {plasmid_count}")
    print(f"Skipped assemblies (missing) : {len(skipped)}")
    if skipped:
        print("----------------------------")
        print("⚠️ Skipped accessions:")
        for acc in skipped:
            print(f"   - {acc}")
    print("============================")

# =========================
# 4. Script Execution
# =========================
if __name__ == "__main__":
    main()


In [ ]:
import re
import pandas as pd
from pathlib import Path
from Bio import SeqIO
from typing import Optional

# =========================
# 1. Configuration
# =========================
base_dir = Path("/media/bahaa/Data/Work/2026/Aeromonas_type_material_23022026/Aeromonas_type_material_23022026/ncbi_dataset/data")
summary_file = base_dir / "data_summary.tsv"
accession_data_dir = base_dir 

output_dir = Path.cwd() / "raw_data"
chr_dir = output_dir / "chromosomes"
plasmid_dir = output_dir / "plasmids"

GENUS_ABBR: Optional[str] = "A"

# =========================
# 2. Helper Functions
# =========================
def sanitize_filename(name: str) -> str:
    """
    Cleans the filename:
    - Replaces hyphens with underscores.
    - Replaces non-alphanumeric (except dots) with underscores.
    - Collapses multiple underscores into one.
    """
    # Replace hyphens with underscores first
    name = name.replace("-", "_")
    # Replace other special characters
    name = re.sub(r'[^A-Za-z0-9_.]', '_', name)
    # Collapse multiple underscores into one
    name = re.sub(r'_+', '_', name)
    return name.strip("_")

def format_organism_name(organism: str, abbr: Optional[str] = None) -> str:
    """Formats organism name, replacing subsp., removing isolate, and handling spacing."""
    # Remove 'isolate' and fix 'subsp.'
    organism = re.sub(r'\bisolate\b', '', organism, flags=re.IGNORECASE)
    organism = organism.replace("subsp.", "sub")
    
    parts = organism.split()
    if abbr and len(parts) >= 2:
        # Keep abbreviation and join the rest with underscores
        rest = "_".join(parts[1:])
        return f"{abbr}_{rest}"
    
    return "_".join(parts)

# =========================
# 3. Main Processing Logic
# =========================
def main():
    print(f"Creating output directories in: {output_dir}")
    chr_dir.mkdir(parents=True, exist_ok=True)
    plasmid_dir.mkdir(parents=True, exist_ok=True)

    try:
        df = pd.read_csv(summary_file, sep="\t")
    except FileNotFoundError:
        print(f"❌ ERROR: Metadata file not found at '{summary_file}'")
        return

    mapping, skipped = [], []
    chromosome_count = plasmid_count = assemblies_found = 0
    used_names = set()

    print("🚀 Starting genome processing...")

    for _, row in df.iterrows():
        accession = row["Assembly Accession"]
        organism = row["Organism Scientific Name"]
        
        # Format the base organism name
        organism_formatted = format_organism_name(organism, GENUS_ABBR)
        
        # Clean the strain info
        strain_raw = str(row.get("Organism Qualifier", ""))
        strain = strain_raw.replace("strain:", "").split(";")[0].strip()
        # Remove 'isolate' from strain if present
        strain = re.sub(r'\bisolate\b', '', strain, flags=re.IGNORECASE).strip()
        
        if strain and strain.lower() != "nan" and strain not in organism_formatted:
            new_name = f"{organism_formatted}_{strain}"
        else:
            new_name = organism_formatted
            
        # Final pass for hyphens, multiple underscores, and illegal chars
        new_name = sanitize_filename(new_name)
        
        if new_name in used_names:
            new_name = f"{new_name}_{accession}"
        used_names.add(new_name)

        asm_dir = accession_data_dir / accession
        try:
            fna_file = next(asm_dir.glob("*_genomic.fna"))
            assemblies_found += 1
        except StopIteration:
            print(f"⚠️ Skipping {accession}, file not found in '{asm_dir}'.")
            skipped.append(accession)
            continue
            
        chr_records, plasmid_records = [], []
        for record in SeqIO.parse(fna_file, "fasta"):
            if "plasmid" in record.description.lower():
                plasmid_records.append(record)
            else:
                chr_records.append(record)

        if chr_records:
            chr_out_path = chr_dir / f"{new_name}.fna"
            for rec in chr_records:
                rec.id = new_name
                rec.description = ""
            SeqIO.write(chr_records, chr_out_path, "fasta")
            mapping.append([accession, new_name, "chromosome", str(chr_out_path)])
            chromosome_count += 1

        if plasmid_records:
            plasmid_out_path = plasmid_dir / f"{new_name}.fna"
            for rec in plasmid_records:
                rec.id = new_name
                rec.description = ""
            SeqIO.write(plasmid_records, plasmid_out_path, "fasta")
            mapping.append([accession, new_name, "plasmid", str(plasmid_out_path)])
            plasmid_count += 1

    map_file = output_dir / "accession_to_newnames.tsv"
    pd.DataFrame(mapping, columns=["accession", "new_name", "type", "file_path"]).to_csv(
        map_file, sep="\t", index=False
    )
    
    print(f"\n📑 Mapping file saved: {map_file}")
    print("\n\n===== 📊 SUMMARY REPORT =====")
    print(f"Total assemblies: {len(df)} | Found: {assemblies_found}")
    print(f"Chromosomes: {chromosome_count} | Plasmids: {plasmid_count}")
    print("============================")

if __name__ == "__main__":
    main()

In [ ]:
%%bash
#!/bin/bash

# ==========================================
# Automated dRep
# ==========================================

# Exit immediately if a command exits with a non-zero status.
set -e  

echo "--- Step 1: Setting a non-interactive matplotlib backend ---"
# This command fixes the error by forcing matplotlib to use a backend
# that doesn't require a graphical user interface.
export MPLBACKEND=Agg

echo "--- Step 2: Creating output directory for dRep results ---"
# Define output directories as variables for clarity
CHR_OUT="1_drep_result/chr"
PLASMID_OUT="1_drep_result/plasmid"
mkdir -p "$CHR_OUT"
mkdir -p "$PLASMID_OUT"

echo "--- Step 3: Activating Conda environment ---"
# Dynamically find the conda installation path
CONDA_BASE=$(conda info --base)
CONDA_SCRIPT="$CONDA_BASE/etc/profile.d/conda.sh"

if [ -f "$CONDA_SCRIPT" ]; then
    source "$CONDA_SCRIPT"
    conda activate drep
else
    echo "❌ Error: Conda activation script not found at '$CONDA_SCRIPT'"
    exit 1
fi

echo "--- Step 4: Running dRep on chromosomes ---"
# Corrected the output path to match the directory created above
dRep dereplicate "$CHR_OUT" -g ./raw_data/chromosomes/*.fna

echo "--- Step 5: Running dRep on plasmids ---"
#dRep dereplicate "$PLASMID_OUT" -g ./raw_data/plasmids/*.fna

echo "--- Deactivating Conda environment ---"
conda deactivate

echo "✅ dRep processing complete!"


Bakta

In [ ]:

%%bash
# ==============================================================================
# 🧬 AUTOMATED BAKTA ANNOTATION PIPELINE
# ==============================================================================
# DESCRIPTION:
# This script automates the high-throughput annotation of bacterial genomes 
# using Bakta. It performs the following steps:
#   1. Scans raw data for .fna chromosome files.
#   2. Filters files based on user-defined INCLUDE/EXCLUDE patterns.
#   3. Generates a UNIQUE LOCUS TAG for each genome based on its taxonomy 
#      and strain ID to ensure downstream compatibility (e.g., with Panaroo).
#   4. Executes Bakta with metadata (Genus, Species, Strain) extraction.
#   5. Produces a summary report of successful and failed runs.
# ==============================================================================

# Make script stop only for critical errors
set -o pipefail

echo "--- Step 1: Creating a list of chromosome files ---"
# Navigate to the data directory or exit if it doesn't exist.
cd raw_data/chromosomes || { echo "❌ Directory not found: raw_data/chromosomes"; exit 1; }

# Create a list of all .fna files.
find . -type f -name "*.fna" -printf "%f\n" > ../raw_fasta_list

# Return to the original directory.
cd ../..

echo "--- Step 2: Creating output directory for Bakta results ---"
mkdir -p 1_bakta_result

echo "--- Step 3: Activating Conda environment ---"
CONDA_SCRIPT="/home/bahaa/anaconda3/etc/profile.d/conda.sh"
if [ -f "$CONDA_SCRIPT" ]; then
    source "$CONDA_SCRIPT"
    conda activate bakta || echo "⚠️ Conda activation failed, continuing with current environment..."
else
    echo "⚠️ Conda script not found at $CONDA_SCRIPT, continuing with current environment..."
fi

# Set environment variables to prevent GUI-related errors and suppress warnings.
export MPLBACKEND=Agg
export PYTHONWARNINGS="ignore"

echo "--- Step 4: Preparing selected list based on filter ---"

# --- Filtering Configuration ---
FILTER_MODE="INCLUDE"  # Options: "INCLUDE", "EXCLUDE", ""
PATTERNS=""

# --- Filtering Logic ---
total_files=$(wc -l < ./raw_data/raw_fasta_list)

if [ -n "$FILTER_MODE" ] && [ -n "$PATTERNS" ]; then
    if [ "$FILTER_MODE" = "INCLUDE" ]; then
        echo "Filtering to INCLUDE only files matching: $PATTERNS"
        grep -E "$PATTERNS" ./raw_data/raw_fasta_list > ./raw_data/filtered_list
    elif [ "$FILTER_MODE" = "EXCLUDE" ]; then
        echo "Filtering to EXCLUDE files matching: $PATTERNS"
        grep -Ev "$PATTERNS" ./raw_data/raw_fasta_list > ./raw_data/filtered_list
    else
        echo "⚠️ Invalid FILTER_MODE: '$FILTER_MODE'. Processing all files."
        cp ./raw_data/raw_fasta_list ./raw_data/filtered_list
    fi
else
    echo "No filter applied. Processing all files."
    cp ./raw_data/raw_fasta_list ./raw_data/filtered_list
fi

selected_files=$(wc -l < ./raw_data/filtered_list)
echo "📄 Total files found: $total_files"
echo "✅ Selected for processing: $selected_files"

# Prepare counters and summary file
processed_count=0
failed_count=0
processed_list="./1_bakta_result/processed_summary.txt"
> "$processed_list"

echo "--- Step 5: Running Bakta on selected chromosome files ---"

while IFS= read -r x; do
    echo "----------------------------------------------------"
    echo "🧬 Processing: $x"

    # Extract metadata from filename
    base_name=$(echo "$x" | sed -e 's/\.fna$//' -e 's/_chr$//')
    genus=$(echo "$base_name" | cut -d "_" -f 1)
    species=$(echo "$base_name" | cut -d "_" -f 2)
    strain=$(echo "$base_name" | cut -d "_" -f 3-)

    # --- UNIQUE LOCUS TAG GENERATION ---
    # Extracts first letter of Genus, first 2 of Species, and last 4 of Strain.
    # We use 'tr' to remove any dots or special characters for the tag.
    G_PRE=$(echo "$genus" | cut -c1 | tr '[:lower:]' '[:upper:]')
    S_PRE=$(echo "$species" | cut -c1-2 | tr '[:lower:]' '[:upper:]')
    STR_ID=$(echo "$strain" | tr -dc '[:alnum:]' | tail -c 4)
    LOCUS_TAG="${G_PRE}${S_PRE}_${STR_ID}"
    
    echo "🏷️  Assigned Locus Tag: $LOCUS_TAG"

    # Define input and output paths
    input_file="./raw_data/chromosomes/$x"
    output_dir="./1_bakta_result/bakta-${genus}_${species}_${strain}"

    # Run Bakta command
    if bakta --db ~/db "$input_file" \
             --output "$output_dir" \
             --genus "$genus" \
             --species "$species" \
             --strain "$strain" \
             --locus-tag "$LOCUS_TAG"; then
        echo "${genus}_${species}_${strain} [Locus: $LOCUS_TAG]" >> "$processed_list"
        ((processed_count++))
    else
        echo "❌ Failed: $x" >&2
        ((failed_count++))
    fi
done < ./raw_data/filtered_list

# --- Summary ---
echo
echo "=========================================="
echo "🎉 Bakta Processing Summary"
echo "=========================================="
echo "📦 Total .fna files:         $total_files"
echo "🧩 Selected for processing:  $selected_files"
echo "✅ Successfully processed:   $processed_count"
echo "⚠️  Failed:                  $failed_count"
echo "📘 Summary saved to:         $processed_list"
echo "=========================================="

Bakta last with very unique locus tag

In [ ]:
%%bash
# ==============================================================================
# 🧬 AUTOMATED BAKTA ANNOTATION PIPELINE (FIXED GRAPHICS BACKEND)
# ==============================================================================

# --- Step 1: File Discovery ---
cd raw_data/chromosomes || { echo "❌ raw_data/chromosomes not found"; exit 1; }
find . -type f -name "*.fna" -printf "%f\n" > ../raw_fasta_list
cd ../..

mkdir -p 1_bakta_result

# --- Step 2: Conda Environment Setup ---
CONDA_SCRIPT="/home/bahaa/anaconda3/etc/profile.d/conda.sh"
if [ -f "$CONDA_SCRIPT" ]; then
    source "$CONDA_SCRIPT"
    conda activate bakta
fi

# --- 🚀 FIX: FORCE NON-INTERACTIVE GRAPHICS BACKEND ---
# This resolves the ValueError: Key backend error by forcing matplotlib 
# to use 'Agg' which works in headless/server environments.
export MPLBACKEND=Agg
export PYTHONWARNINGS="ignore"

# --- Step 3: Filtering Configuration ---
FILTER_MODE="INCLUDE"
PATTERNS="ALL"

if [ -n "$FILTER_MODE" ] && [ -n "$PATTERNS" ]; then
    if [ "$FILTER_MODE" = "INCLUDE" ]; then
        grep -E "$PATTERNS" ./raw_data/raw_fasta_list > ./raw_data/filtered_list
    else
        grep -Ev "$PATTERNS" ./raw_data/raw_fasta_list > ./raw_data/filtered_list
    fi
else
    cp ./raw_data/raw_fasta_list ./raw_data/filtered_list
fi

selected_files=$(wc -l < ./raw_data/filtered_list)
echo "✅ Selected for processing: $selected_files"

# --- Step 4: High-Throughput Execution ---
counter=1
processed_count=0
failed_count=0
summary_file="./1_bakta_result/processed_summary.txt"
> "$summary_file"

while IFS= read -r x; do
    echo "----------------------------------------------------"
    echo "🧬 Isolating: $x"

    # Metadata extraction
    base_name=$(echo "$x" | sed -e 's/\.fna$//' -e 's/_chr$//')
    genus=$(echo "$base_name" | cut -d "_" -f 1)
    species=$(echo "$base_name" | cut -d "_" -f 2)
    strain=$(echo "$base_name" | cut -d "_" -f 3-)

    # UNIQUE LOCUS TAG GENERATION
    # Uses Genus first letter + unique 4-digit ID
    GENUS_PRE=$(echo "$genus" | cut -c1 | tr '[:lower:]' '[:upper:]')
    LOCUS_TAG=$(printf "%s%04d" "$GENUS_PRE" "$counter")
    
    echo "🏷️ Assigned Tag: $LOCUS_TAG"

    input_file="./raw_data/chromosomes/$x"
    output_dir="./1_bakta_result/bakta-${genus}_${species}_${strain}"

    # Run Bakta 
    # Added --skip-plot if you want to be 100% sure the graphics error is bypassed
    if bakta --db ~/db "$input_file" \
             --output "$output_dir" \
             --genus "$genus" \
             --species "$species" \
             --strain "$strain" \
             --locus-tag "$LOCUS_TAG" \
             --threads 16; then
        
        echo "${genus}_${species}_${strain} [Locus: $LOCUS_TAG]" >> "$summary_file"
        ((processed_count++))
        ((counter++)) 
    else
        echo "❌ FAILED: $x" >&2
        ((failed_count++))
        # Optional: counter++ here if you want to skip the ID number for failed runs
    fi
done < ./raw_data/filtered_list

# --- Step 5: Final Summary ---
echo
echo "=========================================="
echo "🎉 Bakta Pipeline Execution Complete"
echo "=========================================="
echo "✅ Successfully processed:   $processed_count"
echo "⚠️ Failed runs:             $failed_count"
echo "📘 Metadata summary:        $summary_file"
echo "=========================================="

Bakta improved

In [ ]:
%%bash
# ==============================================================================
# 🧬 AUTOMATED BAKTA ANNOTATION PIPELINE (V3: MULTILINE & FILE SUPPORT)
# ==============================================================================

# --- Step 1: File Discovery ---
RAW_LIST="./raw_data/raw_fasta_list"
FILTERED_LIST="./raw_data/filtered_list"
TMP_PATTERN="./raw_data/tmp_patterns.txt"

mkdir -p 1_bakta_result
# Generate list of available fasta files
cd raw_data/chromosomes || { echo "❌ raw_data/chromosomes not found"; exit 1; }
find . -type f -name "*.fna" -printf "%f\n" > "../../$RAW_LIST"
cd ../..

# --- Step 2: Conda & Graphics Fix ---
CONDA_SCRIPT="/home/bahaa/anaconda3/etc/profile.d/conda.sh"
[ -f "$CONDA_SCRIPT" ] && source "$CONDA_SCRIPT" && conda activate bakta
export MPLBACKEND=Agg
export PYTHONWARNINGS="ignore"

# --- Step 3: IMPROVED FILTERING CONFIGURATION ---
FILTER_MODE="INCLUDE" 

# You can now paste your list here directly or use a filename
PATTERNS="A_aquatica_AE235.fna
A_australiensis_CECT_8023.fna
A_bestiarum_CCUG_41859.fna
A_bivalvium_CECT_7113.fna
A_caviae_NCTC12244.fna
A_dhakensis_CIP_107500.fna
A_diversa_CDC_2478_85.fna
A_encheleia_NCTC12917.fna
A_enteropelogenes_FDAARGOS_1509.fna
A_eucrenophila_CCUG_30340.fna
A_finlandensis_4287D.fna
A_fluvialis_LMG_24681.fna
A_hydrophila_sub_hydrophila_ATCC_7966_ATCC_7966.fna
A_jandaei_FDAARGOS_986.fna
A_lusitana_MDC_2473.fna
A_media_CCM_3653_GCF_042647745.1.fna
A_popoffii_CIP_105493.fna
A_rivipollensis_DSM_24593.fna
A_rivuli_DSM_22539.fna
A_salmonicida_NCIMB1102.fna
A_sanarellii_LMG_24682.fna
A_schubertii_CECT_4240.fna
A_sobria_CIP_74.33.fna
A_taiwanensis_LMG_24683.fna
A_tecta_CECT_7082.fna
A_veronii_CECT_4486.fna
A_mytilicola_sub_aquatica_A_8.fna
A_ichthyocola_A_5.fna
A_piscicola_LMG_24783.fna
A_oralensis_AB005.fna
A_lacus_AE122.fna"

# Logic to handle File vs Multiline String vs Single Pattern
if [ -f "$PATTERNS" ]; then
    echo "📄 Using existing file: $PATTERNS"
    grep_input="$PATTERNS"
else
    # Save the string to a temporary file to avoid "Unmatched (" shell errors
    echo "$PATTERNS" > "$TMP_PATTERN"
    grep_input="$TMP_PATTERN"
    echo "🔍 Using provided patterns/list..."
fi

if [ "$FILTER_MODE" = "INCLUDE" ]; then
    # Use -F (fixed strings) and -f (read patterns from file) for safety
    grep -Fxf "$grep_input" "$RAW_LIST" > "$FILTERED_LIST"
else
    grep -Fvxf "$grep_input" "$RAW_LIST" > "$FILTERED_LIST"
fi

selected_files=$(wc -l < "$FILTERED_LIST")
echo "✅ Selected for processing: $selected_files"

# --- Step 4: High-Throughput Execution ---
counter=1
summary_file="./1_bakta_result/processed_summary.txt"
> "$summary_file"

while IFS= read -r x; do
    [ -z "$x" ] && continue
    echo "----------------------------------------------------"
    echo "🧬 ($counter/$selected_files) Processing: $x"

    base_name=$(echo "$x" | sed 's/\.fna$//')
    genus=$(echo "$base_name" | cut -d "_" -f 1)
    species=$(echo "$base_name" | cut -d "_" -f 2)
    strain=$(echo "$base_name" | cut -d "_" -f 3-)

    GENUS_PRE=$(echo "$genus" | cut -c1 | tr '[:lower:]' '[:upper:]')
    LOCUS_TAG=$(printf "%s%04d" "$GENUS_PRE" "$counter")
    
    # Use absolute or relative path correctly
    bakta --db ~/db "./raw_data/chromosomes/$x" \
          --output "./1_bakta_result/bakta-${base_name}" \
          --genus "$genus" \
          --species "$species" \
          --strain "${strain:-unknown}" \
          --locus-tag "$LOCUS_TAG" \
          --threads 16 --skip-plot
    
    if [ $? -eq 0 ]; then
        echo "$base_name | $LOCUS_TAG | Success" >> "$summary_file"
    else
        echo "❌ FAILED: $x" >> "$summary_file"
    fi
    ((counter++))
done < "$FILTERED_LIST"

Improved ANI

In [ ]:
%%bash
#!/bin/bash

# ==============================================================================
# 🧬 AUTOMATED PYANI PIPELINE (FIXED FILTERING)
# ==============================================================================

# --- Configuration Area ---

# The script now looks for these keywords ANYWHERE in the filename.
INCLUDE_LIST_RAW="
A_australiensis_CECT_8023.fna
A_bestiarum_CCUG_41859.fna
A_caviae_NCTC12244.fna
A_dhakensis_CIP_107500.fna
A_diversa_CDC_2478_85.fna
A_encheleia_NCTC12917.fna
A_enteropelogenes_FDAARGOS_1509.fna
A_eucrenophila_CCUG_30340.fna
A_hydrophila_sub_hydrophila_ATCC_7966_ATCC_7966.fna
A_jandaei_FDAARGOS_986.fna
A_lusitana_MDC_2473.fna
A_media_CCM_3653_GCF_042647745.1.fna
A_rivipollensis_DSM_24593.fna
A_rivuli_DSM_22539.fna
A_salmonicida_NCIMB1102.fna
A_schubertii_CECT_4240.fna
A_sobria_CIP_74.33.fna
A_taiwanensis_LMG_24683.fna
A_veronii_CECT_4486.fna
A_mytilicola_sub_aquatica_A_8.fna
A_ichthyocola_A_5.fna
A_oralensis_AB005.fna
"

EXCLUDE_LIST_RAW=""

METHODS=("ANIm" "ANIb") 
INPUT_DIR="raw_data/chromosomes"
BASE_OUT_DIR="1_pyani"

# --- Step 1: Workspace & Pattern Setup ---
RUN_NAME="Analysis_$(date +%Y%m%d_%H%M)"
FINAL_OUTPUT_DIR="$BASE_OUT_DIR/$RUN_NAME"
TEMP_DATA_DIR="$BASE_OUT_DIR/temp_data_$RUN_NAME"

mkdir -p "$FINAL_OUTPUT_DIR" "$TEMP_DATA_DIR"

# Clean lists
INC_CLEAN=$(echo "$INCLUDE_LIST_RAW" | tr -d '\r' | grep -v '^$')
INCLUDE_PATTERN=$(echo "$INC_CLEAN" | tr '\n' '|' | sed 's/|$//')
EXCLUDE_PATTERN=$(echo "$EXCLUDE_LIST_RAW" | tr -d '\r' | grep -v '^$' | tr '\n' '|' | sed 's/|$//')

echo "📂 Filtering genomes in $INPUT_DIR..."
count=0
for file in "$INPUT_DIR"/*.fna; do
    [ -f "$file" ] || continue
    filename=$(basename "$file")
    
    # 1. Check Exclusion first (Matches exactly if listed)
    if echo "$filename" | grep -Eq "^($EXCLUDE_PATTERN)$"; then
        continue
    fi

    # 2. Check Inclusion (Match ANYWHERE in the name)
    if [[ "$INC_CLEAN" != "ALL" ]]; then
        # Removed the '^' anchor to allow matching middle of filename
        if ! echo "$filename" | grep -Eq "($INCLUDE_PATTERN)"; then
            continue
        fi
    fi

    ln -sf "$(realpath "$file")" "$TEMP_DATA_DIR/$filename"
    echo "✅ Added: $filename"
    ((count++))
done

echo "📊 Total Genomes Included: $count"

if [ "$count" -lt 2 ]; then
    echo "❌ ERROR: pyani requires at least 2 files. Found: $count"
    exit 1
fi

# --- Step 2: Loop Through Methods ---
# [Logic remains same as previous working version]
CONDA_SCRIPT="/home/bahaa/anaconda3/etc/profile.d/conda.sh"
if [ -f "$CONDA_SCRIPT" ]; then
    source "$CONDA_SCRIPT"
    conda activate pyani_env || echo "⚠️ Conda activate failed."
fi

for M in "${METHODS[@]}"; do
    echo "🧬 Running Method: $M..."
    if average_nucleotide_identity.py -i "$TEMP_DATA_DIR" -o "$FINAL_OUTPUT_DIR/${M}_output" -m "$M" -g; then
        echo "✅ $M success."
        # Python reporting logic (Standard matplotlib 'Agg' backend included)
        python3 <<EOF
import pandas as pd
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

method = "$M"
ani_path = f"$FINAL_OUTPUT_DIR/{method}_output/{method}_percentage_identity.tab"
if os.path.exists(ani_path):
    df = pd.read_csv(ani_path, sep='\t', index_col=0)
    report = df.stack().reset_index()
    report.columns = ['Genome_A', 'Genome_B', 'ANI']
    report = report[report['Genome_A'] != report['Genome_B']]
    report['Status'] = report['ANI'].apply(lambda x: 'SAME' if x >= 0.95 else 'DIFF')
    report.to_csv(f"$FINAL_OUTPUT_DIR/{method}_report.csv", index=False)
    
    if len(df) > 1:
        plt.figure(figsize=(10, 8))
        dendrogram(linkage(1-df, 'average'), labels=df.index, leaf_rotation=90)
        plt.axhline(y=0.05, color='r', linestyle='--')
        plt.savefig(f"$FINAL_OUTPUT_DIR/{method}_dendrogram.png")
EOF
    else
        echo "⚠️  $M failed."
    fi
done

rm -rf "$TEMP_DATA_DIR"
echo "🏁 Done! Folder: $FINAL_OUTPUT_DIR"

In [ ]:
%%bash
#!/bin/bash

# ==============================================================================
# 🧬 ROARY PAN-GENOME PIPELINE (CUSTOM IDENTITY & FILTERED)
# ==============================================================================

# --- Configuration Area ---
IDENTITY_THRESHOLD=70      # Minimum Blastp identity for ortholog clustering
THREADS=16

INCLUDE_LIST_RAW="A_australiensis_CECT_8023
A_bestiarum_CCUG_41859
A_caviae_NCTC12244
A_dhakensis_CIP_107500
A_diversa_CDC_2478_85
A_encheleia_NCTC12917
A_enteropelogenes_FDAARGOS_1509
A_eucrenophila_CCUG_30340
A_hydrophila_sub_hydrophila_ATCC_7966_ATCC_7966
A_jandaei_FDAARGOS_986
A_lusitana_MDC_2473
A_media_CCM_3653_GCF_042647745.1
A_rivipollensis_DSM_24593
A_rivuli_DSM_22539
A_salmonicida_NCIMB1102
A_schubertii_CECT_4240
A_sobria_CIP_74.33
A_taiwanensis_LMG_24683
A_veronii_CECT_4486
A_mytilicola_sub_aquatica_A_8
A_ichthyocola_A_5
A_oralensis_AB005"

EXCLUDE_LIST_RAW=""

BAKTA_DIR="./1_bakta_result"
BASE_ROARY_DIR="./1_Roary"
# --------------------------

# --- Step 1: Workspace Setup ---
RUN_NAME="PanGenome_I${IDENTITY_THRESHOLD}_$(date +%Y%m%d_%H%M)"
FINAL_OUT_DIR="$BASE_ROARY_DIR/$RUN_NAME"
GFF_DIR="$FINAL_OUT_DIR/gff"

mkdir -p "$GFF_DIR"

# Prepare Patterns
INC_PATTERN=$(echo "$INCLUDE_LIST_RAW" | tr -d '\r' | grep -v '^$' | tr '\n' '|' | sed 's/|$//')
EXC_PATTERN=$(echo "$EXCLUDE_LIST_RAW" | tr -d '\r' | grep -v '^$' | tr '\n' '|' | sed 's/|$//')

echo "📂 Filtering Bakta GFFs into $GFF_DIR..."
count=0
for gff_path in "$BAKTA_DIR"/*/*.gff3; do
    filename=$(basename "$gff_path")
    if [[ -n "$EXC_PATTERN" ]] && echo "$filename" | grep -Eq "($EXC_PATTERN)"; then continue; fi
    if [[ "$INCLUDE_LIST_RAW" != *"ALL"* ]] && ! echo "$filename" | grep -Eq "($INC_PATTERN)"; then continue; fi

    # Roary compatibility: rename .gff3 to .gff
    cp "$gff_path" "$GFF_DIR/${filename%.gff3}.gff"
    ((count++))
done

echo "📊 Isolates included: $count (Identity Threshold: ${IDENTITY_THRESHOLD}%)"

# --- Step 2: Run Roary ---
echo "🧬 Running Roary..."
# -i flag sets the minimum percentage identity for blastp
roary -f "$FINAL_OUT_DIR/roary_results" -e -n -v -p $THREADS -i $IDENTITY_THRESHOLD -r -g 100000 -o clustered_proteins "$GFF_DIR"/*.gff

cd "$FINAL_OUT_DIR/roary_results"

# --- Step 3: Phylogeny & Post-Processing ---
echo "🌳 Building Core Genome Tree..."
fasttreeMP -nt -gtr core_gene_alignment.aln > core_genome_tree.newick

echo "🎨 Generating Presence/Absence Plots..."
python /home/bahaa/bio/soft/Roary-master/contrib/roary_plots/roary_plots.py core_genome_tree.newick gene_presence_absence.csv

# Standardize formats for PanPG
echo "📝 Formatting outputs..."
/home/bahaa/bio/soft/remove_columns.sh ./gene_presence_absence.csv ./gene_presence_absence_clean.csv
/home/bahaa/bio/soft/csv_to_tsv_fill_empty_v2.sh ./gene_presence_absence_clean.csv ./gene_presence_absence.tsv --tab

# --- Step 4: Core SNP Analysis ---
source /home/bahaa/anaconda3/etc/profile.d/conda.sh
conda activate snp-site
echo "🧬 Extracting Core SNPs..."
snp-sites -mvph -o CORE_SNPS core_gene_alignment.aln
fasttreeMP -nt -gtr CORE_SNPS.snp_sites.aln > core_snp_tree.newick

echo "🏁 Pan-genome analysis complete!"
echo "📂 Results folder: $FINAL_OUT_DIR/roary_results"

In [ ]:
%%bash
#!/bin/bash

# ==============================================================================
# 🧬 NEAT DOWNSTREAM ANALYSIS PIPELINE
# ==============================================================================

# --- Configuration Area ---
GENE_LIST="my_genes.txt"
BAKTA_FFN_DIR="../../2_bakta"
REF_TREE="core_genome_tree.newick"
THREADS=16

# Create a master output directory
TIMESTAMP=$(date +%Y%m%d_%H%M)
OUT="Downstream_Analysis_$TIMESTAMP"
mkdir -p "$OUT/1_locus_tags" "$OUT/2_extracted_genes" "$OUT/3_renamed_genes" \
         "$OUT/4_alignments" "$OUT/5_gene_trees_nwk" "$OUT/6_gene_trees_pdf"

# --- Step 1: Locus Tag Extraction (Python) ---
cat << 'EOF' > extract_tags.py
import pandas as pd
import sys, os
def extract_tags(input_list, csv_file='gene_presence_absence.csv', out_dir=''):
    df = pd.read_csv(csv_file, low_memory=False)
    with open(input_list, 'r') as f: genes = [line.strip() for line in f if line.strip()]
    metadata = ['Gene','Non-unique Gene name','Annotation','No. isolates','No. sequences','Avg sequences per isolate','Genome Fragment','Order within Fragment','Accession','Specific Gene Groups','Average Length (bp)','Compound Gene Name']
    strains = [c for c in df.columns if c not in metadata]
    for target in genes:
        row = df[df['Gene'] == target]
        if row.empty: continue
        tags = [t for col in strains for t in str(row[col].values[0]).split() if pd.notna(row[col].values[0]) and not t.replace('.','').isdigit()]
        if tags:
            with open(f"{out_dir}/{target}_tags.txt", 'w') as f: f.write("\n".join(tags))
extract_tags(sys.argv[1], out_dir=sys.argv[2])
EOF

python3 extract_tags.py "$GENE_LIST" "$OUT/1_locus_tags"

# --- Step 2: Sequence Extraction ---
echo "📂 Extracting sequences..."
for gene in $(cat "$GENE_LIST"); do
    TAG_FILE="$OUT/1_locus_tags/${gene}_tags.txt"
    if [ -f "$TAG_FILE" ]; then
        seqkit grep -f "$TAG_FILE" "$BAKTA_FFN_DIR"/*/*.ffn -o "$OUT/2_extracted_genes/${gene}.fa"
    fi
done

# --- Step 3: Rename Sequences (Python) ---
cat << 'EOF' > rename_fasta.py
import pandas as pd
import os, sys, glob
def rename_fasta(in_dir, out_dir, csv_file='gene_presence_absence.csv'):
    df = pd.read_csv(csv_file, low_memory=False)
    metadata = ['Gene','Non-unique Gene name','Annotation','No. isolates','No. sequences','Avg sequences per isolate','Genome Fragment','Order within Fragment','Accessory Fragment','Accessory Order with Fragment','QC','Min group size nuc','Max group size nuc','Avg group size nuc']
    strains = [c for c in df.columns if c not in metadata]
    mapping = {}
    for s in strains:
        col_data = df[s].dropna().astype(str)
        for entry in col_data:
            for item in entry.replace('\t', ' ').split():
                mapping[item.strip()] = s
    for f in glob.glob(f"{in_dir}/*.fa"):
        out_f = os.path.join(out_dir, os.path.basename(f))
        with open(f,'r') as fin, open(out_f,'w') as fout:
            for line in fin:
                if line.startswith('>'):
                    tag = line.strip().split()[0][1:]
                    name = mapping.get(tag, tag).replace(".","_").replace("-","_").replace(" ","_")
                    fout.write(f">{name}\n")
                else: fout.write(line)
rename_fasta(sys.argv[1], sys.argv[2])
EOF

python3 rename_fasta.py "$OUT/2_extracted_genes" "$OUT/3_renamed_genes"

# --- Step 4: Alignment & Tree Construction ---

echo "🧬 Aligning and building trees..."
ls "$OUT/3_renamed_genes"/*.fa | parallel -j $THREADS \
    "mafft --auto {} > $OUT/4_alignments/{/.}_aligned.fasta 2>/dev/null"

for f in "$OUT/4_alignments"/*_aligned.fasta; do
    base=$(basename "$f" _aligned.fasta)
    fasttree -nt -gtr "$f" 2>/dev/null | sed "s/'//g" > "$OUT/5_gene_trees_nwk/${base}.nwk"
    
    Rscript -e "library(ape); t<-read.tree('$OUT/5_gene_trees_nwk/${base}.nwk'); \
    pdf('$OUT/6_gene_trees_pdf/${base}.pdf'); \
    plot(t, main='Phylogeny: $base', cex=0.7, no.margin=T); \
    add.scale.bar(); dev.off()" 2>/dev/null
done

# --- Step 5: Congruence Scoring (R) ---

cat << 'EOF' > congruence.R
library(ape)
args <- commandArgs(trailingOnly = TRUE)
out_dir <- args[1]
ref_tree <- args[2]

clean <- function(l) gsub("[. -]", "_", l)
ref <- read.tree(ref_tree)
ref$tip.label <- clean(ref$tip.label)
files <- list.files(paste0(out_dir, "/5_gene_trees_nwk"), pattern="\\.nwk$", full.names=T)
results <- data.frame(Gene=basename(files), RF_Dist=NA, Congruence_Score=NA)

for(i in 1:length(files)){
    tree <- read.tree(files[i])
    tree$tip.label <- clean(tree$tip.label)
    common <- intersect(tree$tip.label, ref$tip.label)
    if(length(common) >= 4){
        p_gene <- multi2di(unroot(keep.tip(tree, common)))
        p_ref <- multi2di(unroot(ref, common))
        rf <- dist.topo(p_gene, p_ref)
        results$RF_Dist[i] <- rf
        results$Congruence_Score[i] <- 1 - (rf / (2 * (length(common) - 3)))
    }
}
write.csv(results, paste0(out_dir, "/congruence_summary.csv"), row.names=F)
EOF

Rscript congruence.R "$OUT" "$REF_TREE"

# Clean up temp scripts
rm extract_tags.py rename_fasta.py congruence.R

echo "-----------------------------------------------"
echo "🏁 Done! All clean results are in: $OUT"
echo "Summary file: $OUT/congruence_summary.csv"